In [401]:
from rasterio.features import shapes
from shapely.geometry import shape
import time
from rasterio.features import rasterize
import requests
import os
import pystac
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import rioxarray
import torch
import torch.nn.functional as F

import hvplot.xarray
import hvplot.pandas

from mapminer import miners
from ultralytics import YOLO
from mapminer.models import DiNOV3
from shapely.geometry import Polygon, LineString, Point, box

from scipy.ndimage import binary_dilation
from sklearn.decomposition import PCA
from rasterio.enums import Resampling


def overpass(q, tries=4):
    urls = ['https://overpass-api.de/api/interpreter', 'https://overpass.kumi.systems/api/interpreter',
            'https://overpass.osm.jp/api/interpreter']
    for i in range(tries):
        try:
            r = requests.post(urls[i % len(urls)], data={'data':q},
                              headers={'User-Agent':'nepalflood-research/1.0'}, timeout=90)
            if r.status_code == 200 and r.text.lstrip().startswith('{'):
                return r.json()['elements']
        except Exception:
            pass
        time.sleep(4 * (i + 1))
    print('  overpass unavailable, continuing without OSM')
    return []


In [428]:
#lat, lon = 28.1620, 85.3380      # Syabrubesi  - 8 scenes, river, no infra
#lat, lon = 28.2790, 85.3780      # Rasuwagadhi - 7 scenes, river + dam
#lat, lon = 28.1120, 85.2960      # Dhunche     - NO WATER, skipped

lat, lon = 28.18384, 85.30399   
size = 1024
res = 0.6
radius = size * res / 2

flood_date = '2026-08-26'
event = 'https://vantor-opendata.s3.amazonaws.com/events/Nepal-Flooding-Aug-2026'

<br>**Google Basemap : Pre-Flood Reference (Vantor archive over the corridor is 2021/2023 only)**

In [ ]:
ds_pre = miners.GoogleBaseMapMiner().fetch(lat=lat, lon=lon, radius=radius, resolution=res)
ds_pre = ds_pre.isel(band=range(3)) if 'band' in ds_pre.dims else ds_pre
ds_pre = ds_pre.isel(y=slice(0, size), x=slice(0, size)).compute()

In [ ]:
ds_pre.hvplot.rgb(x='x',y='y',bands='band',height=600,width=600,rasterize=True,robust=True,title='Google Basemap : Pre Flood')

BokehModel(combine_events=True, render_bundle={'docs_json': {'3a7ab17b-676a-4595-91ca-5e14b78d1bf5': {'version…

<br>**Vantor Open Data : Post-Flood Scenes (Aug 27-28), 8 overlapping collects over the upper corridor**

In [ ]:
catalog = pystac.Collection.from_file(f'{event}/collection.json')

df_items = pd.DataFrame([
    dict(id=i.id, datetime=str(i.datetime.date()), cloud=i.properties.get('eo:cloud_cover'),
         off_nadir=i.properties.get('view:off_nadir'), href=i.assets['visual'].href, bbox=i.bbox)
    for i in catalog.get_items()])

df_items = df_items[df_items.datetime >= flood_date].reset_index(drop=True)
df_items = df_items[df_items.bbox.apply(lambda b: b[0] <= lon <= b[2] and b[1] <= lat <= b[3])].reset_index(drop=True)
df_items

,id,datetime,cloud,off_nadir,href,bbox
0,B030001100CF1110,2026-08-28,81,25.06,https://vantor-opendata.s3.amazonaws.com/event...,"[85.2809355740028, 28.14405021362965, 85.46943..."
1,B030001100CF1310,2026-08-28,78,21.53,https://vantor-opendata.s3.amazonaws.com/event...,"[85.26753834772008, 28.108669765925608, 85.451..."
2,B030001100CF1610,2026-08-28,79,12.23,https://vantor-opendata.s3.amazonaws.com/event...,"[85.27036854144649, 28.11599546216905, 85.4484..."
3,B040001100881410,2026-08-27,73,21.72,https://vantor-opendata.s3.amazonaws.com/event...,"[85.28915546816656, 28.11104018377955, 85.4378..."
4,B040001100881610,2026-08-27,71,9.13,https://vantor-opendata.s3.amazonaws.com/event...,"[85.29508927001986, 28.120836047800015, 85.431..."
5,B040001100882F10,2026-08-27,79,12.34,https://vantor-opendata.s3.amazonaws.com/event...,"[85.26636068176573, 27.946995056909795, 85.412..."
6,B110001101165110,2026-08-28,78,19.45,https://vantor-opendata.s3.amazonaws.com/event...,"[85.31860373688943, 27.546103259468413, 85.442..."


In [ ]:
bounds = ds_pre.rio.transform_bounds('EPSG:4326') if ds_pre.rio.crs != 'EPSG:4326' else ds_pre.rio.bounds()

def fetch_window(href):
    da = rioxarray.open_rasterio(href, chunks={'x':512,'y':512})
    da = da.rio.clip_box(*bounds, crs='EPSG:4326').isel(band=range(3))
    da = da.rio.reproject_match(ds_pre, resampling=Resampling.bilinear)
    return da

assert len(df_items), 'NO_SCENES'
ds_post = xr.concat([fetch_window(h) for h in df_items.href], dim='scene').assign_coords(scene=df_items.id.values).compute()

<br>**Per-Pixel Cloud Rejection : bright + achromatic pixels are cloud, composite the rest**

In [407]:
V = ds_post.max(dim='band').astype('float32')
S = ((ds_post.max(dim='band') - ds_post.min(dim='band')) / ds_post.max(dim='band').where(lambda x: x > 0)).astype('float32')

thr_v, thr_s = 170, 0.15
ds_cloud = ((V > thr_v) & (S < thr_s)) | (ds_post.sum(dim='band') == 0)

ds_valid = (~ds_cloud).sum(dim='scene')
df_cover = pd.DataFrame({'scene': ds_post.scene.values,
                         'cloud_%': (ds_cloud.mean(dim=['x','y']) * 100).round(1).values})
print(df_cover.to_string(index=False))
print(f'\nclear in >=1 scene : {float((ds_valid > 0).mean()) * 100:.1f}%')
print(f'clear in >=3 scene : {float((ds_valid >= 3).mean()) * 100:.1f}%')

           scene  cloud_%
B030001100CF1110      0.2
B030001100CF1310      0.4
B030001100CF1610      0.6
B040001100881410      0.5
B040001100881610      0.8
B040001100882F10      1.2
B110001101165110      1.8

clear in >=1 scene : 100.0%
clear in >=3 scene : 100.0%


In [408]:
ds_comp = ds_post.where(~ds_cloud).median(dim='scene').round().fillna(0).astype('uint8')
ds_comp = ds_comp.where(ds_valid > 0, 0)

In [ ]:
ds_comp.hvplot.rgb(x='x',y='y',bands='band',height=600,width=650,rasterize=True,robust=True,title='Cloud-Free Composite : Post Flood')

In [409]:
ds_comp['x'] = ds_pre['x']
ds_comp['y'] = ds_pre['y']

In [411]:
(ds_pre.hvplot.rgb(x='x',y='y',bands='band',height=500,width=530,rasterize=True,robust=True,title='Pre Flood : Google Basemap')+\
 ds_comp.hvplot.rgb(x='x',y='y',bands='band',height=500,width=530,rasterize=True,robust=True,title='Post Flood : Vantor Composite')).cols(2)

BokehModel(combine_events=True, render_bundle={'docs_json': {'b51815e9-c225-40d1-95ec-2f4024a70dea': {'version…

<br>**DINOv3 : Pre/Post Water Segmentation, Seeded from OSM River**

In [379]:
dino  = DiNOV3(architecture='vit-l-sat').eval().to('mps').half()
patch = dino.model.patch_embed.proj.stride

def embed(ds):
    x = torch.from_numpy(ds.transpose('band','y','x').values[None].astype('float32')/255.).to('mps').half()
    H, W = x.shape[-2:]
    with torch.no_grad(): t = dino(x, tokens=True)              # (B, N, C)
    B, N, C = t.shape
    f = t.permute(0,2,1).reshape(B, C, H//patch[0], W//patch[1])  # transpose, not reshape
    return F.normalize(f, dim=1)[0].float().cpu().numpy()

f_pre, f_post = embed(ds_pre), embed(ds_comp)


In [380]:
cache = f'/tmp/osm_water_{lat:.4f}_{lon:.4f}.gpkg'

if os.path.exists(cache):
    df_water = gpd.read_file(cache)
else:
    b = ds_pre.rio.transform_bounds('EPSG:4326')
    q = f'[out:json][timeout:60];(way({b[1]},{b[0]},{b[3]},{b[2]})["waterway"~"river|stream"];'\
        f'way({b[1]},{b[0]},{b[3]},{b[2]})["natural"="water"];);out geom;'
    geoms = []
    for e in overpass(q):
        pts = [(p['lon'], p['lat']) for p in e.get('geometry', [])]
        if len(pts) < 2: continue
        geoms.append({'geometry': Polygon(pts) if (pts[0]==pts[-1] and len(pts)>3) else LineString(pts),
                      'name': e.get('tags',{}).get('name')})
    df_water = gpd.GeoDataFrame(geoms, columns=['geometry','name'], crs='epsg:4326')
    if len(df_water): df_water.to_file(cache, driver='GPKG')

if len(df_water):
    df_water = df_water.to_crs(ds_comp.rio.crs)
    print(f'osm river : {len(df_water)}  {df_water.geometry.type.value_counts().to_dict()}')
    ds_water_osm = rasterize([(z,1) for z in df_water.buffer(10).geometry], out_shape=ds_pre.shape[-2:],
                             transform=ds_pre.rio.transform(), dtype='uint8')
else:                                                        # no OSM river -> ESRI LULC water
    print('no osm river, falling back to ESRI LULC water class')
    lulc = miners.ESRILULCMiner().fetch(polygon=box(*ds_pre.rio.transform_bounds('EPSG:4326')),
                                       daterange='2024-01-01/2024-12-31')['data'].isel(time=-1).astype('uint8').compute()
    ds_water_osm = ((lulc == 1).astype('uint8').rio.write_crs(lulc.rio.crs)
                    .rio.reproject_match(ds_pre, resampling=Resampling.nearest).values)

ds_water_osm = xr.DataArray(ds_water_osm, dims=['y','x'],
                            coords={'y':ds_pre.y,'x':ds_pre.x}).rio.write_crs(ds_comp.rio.crs)

seed = ds_water_osm.values.astype(bool)[::patch[0], ::patch[1]][:f_post.shape[1], :f_post.shape[2]]
assert seed.sum() > 20, f'seed too small: {seed.sum()} patches'
print(f'seed patches : {seed.sum()} / {seed.size}')

osm river : 4  {'LineString': 3, 'Polygon': 1}
seed patches : 627 / 4096


In [382]:
if False:
    print('skipped: no water in tile')
else:
    c, h, w = f_post.shape
    v = f_post.reshape(c, -1)
    proto = v[:, seed.ravel()].mean(1); proto = proto/np.linalg.norm(proto)

    sim = xr.DataArray((v.T @ proto).reshape(h, w), dims=['y','x'],
                       coords={'y':np.linspace(float(ds_comp.y[0]), float(ds_comp.y[-1]), h),
                               'x':np.linspace(float(ds_comp.x[0]), float(ds_comp.x[-1]), w)}
                      ).rio.write_crs(ds_comp.rio.crs)

    ds_water_dino = (sim > 0.45).astype('uint8').rio.write_crs(ds_comp.rio.crs)

    c, h, w = f_pre.shape
    v = f_pre.reshape(c, -1)
    proto_pre = v[:, seed.ravel()].mean(1); proto_pre = proto_pre/np.linalg.norm(proto_pre)

    sim_pre = xr.DataArray((v.T @ proto_pre).reshape(h, w), dims=['y','x'],
                           coords={'y':np.linspace(float(ds_pre.y[0]), float(ds_pre.y[-1]), h),
                                   'x':np.linspace(float(ds_pre.x[0]), float(ds_pre.x[-1]), w)}
                          ).rio.write_crs(ds_pre.rio.crs)

    ds_water_dino_pre = (sim_pre > 0.60).astype('uint8').rio.write_crs(ds_pre.rio.crs)

    px_d = abs(float(sim.x[1]-sim.x[0])) * abs(float(sim.y[1]-sim.y[0]))
    print(f'dinov3 pre  : {float(ds_water_dino_pre.sum())*px_d/1e6:.4f} km2')
    print(f'dinov3 post : {float(ds_water_dino.sum())*px_d/1e6:.4f} km2')
    print(f'expansion   : {float(ds_water_dino.sum())/max(float(ds_water_dino_pre.sum()),1):.2f}x')


dinov3 pre  : 0.0931 km2
dinov3 post : 0.3011 km2
expansion   : 3.23x


In [383]:
(ds_pre.hvplot.rgb(x='x',y='y',bands='band',height=600,width=650,rasterize=True,robust=True,title='Pre Flood : DINOv3 Water')*\
 ds_water_dino_pre.where(ds_water_dino_pre>0).hvplot(x='x',y='y',rasterize=True,cmap=['cyan'],colorbar=False,alpha=0.5)+\
 ds_comp.hvplot.rgb(x='x',y='y',bands='band',height=600,width=650,rasterize=True,robust=True,title='Post Flood : DINOv3 Water')*\
 ds_water_dino.where(ds_water_dino>0).hvplot(x='x',y='y',rasterize=True,cmap=['red'],colorbar=False,alpha=0.5)).cols(2)

BokehModel(combine_events=True, render_bundle={'docs_json': {'f6e9b4ca-5237-4b9f-bbc2-6efae0850c8e': {'version…

<br>**Building-Miner : YOLOv11-OBB Footprint Detection on Pre/Post**

In [ ]:
model = YOLO('../weights/building-miner-stable.pt')
_ = model.eval()

def get_buildings(ds, device='mps', conf=0.10, iou=0.70, google=True):
    ds = ds.rio.reproject(ds.rio.crs,resolution=0.3)
    ds = ds.transpose('y','x','band').sortby('x').sortby('y')
    imgsz = int(np.ceil(max(ds.shape[:2])/32)*32)
    result = model.predict(ds.data, conf=conf, iou=iou, device=device, verbose=False,
                           imgsz=imgsz, max_det=int(1e8))[0].cpu()

    df = []
    for obb in result.obb:
        coords = obb.xyxyxyxy[0].cpu().numpy().astype('int32')
        coords = np.stack([ds.x.isel(x=np.clip(coords[:,0],0,len(ds.x)-1)),
                           ds.y.isel(y=np.clip(coords[:,1],0,len(ds.y)-1))], axis=1)
        df.append({'confidence':float(obb.conf[0]), 'geometry':Polygon(coords)})

    df = gpd.GeoDataFrame(pd.DataFrame(df), crs=ds.rio.crs) if len(df) else \
         gpd.GeoDataFrame(columns=['confidence','geometry'], crs=ds.rio.crs)

    if google:                                                   # merge google open buildings
        gb = miners.GoogleBuildingMiner().fetch(polygon=box(*ds.rio.transform_bounds('EPSG:4326')))
        gb = gb[['confidence','geometry']].set_crs('epsg:4326', allow_override=True).to_crs(ds.rio.crs)
        gb = gb[gb.intersects(box(*ds.rio.bounds()))].reset_index(drop=True)          # clip to AOI
        df = gpd.GeoDataFrame(pd.concat([df, gb], ignore_index=True), crs=ds.rio.crs)

    df = df[df.area>(10*10)]
    if len(df)==0: return df
    df['geometry'] = df.buffer(0)
    df['area'] = df.geometry.area
    df = df.sort_values('area').reset_index(drop=True)
    for index in df.index[:-1]:
        factor = df.loc[(index+1):].intersection(df.loc[index,'geometry']).area.max()/df.loc[index,'geometry'].area
        if factor > 0.75: df = df.drop(index)
    return df.reset_index(drop=True)

In [419]:
df_pre  = get_buildings(ds_pre,google=False)

(2040, 2040, 3)


In [423]:
df_post = get_buildings(ds_comp, google=False)

(2040, 2040, 3)


In [425]:
df_pre

,confidence,geometry,area,survived
0,0.342991,"POLYGON ((9504084.8 3283241.81, 9504094.1 3283...",174.330,False
1,0.211775,"POLYGON ((9504107 3283069.91, 9504105.8 328305...",187.110,False
2,0.416686,"POLYGON ((9504093.5 3283498.31, 9504103.7 3283...",205.020,False
3,0.100370,"POLYGON ((9504338.9 3283250.51, 9504336.2 3283...",207.720,False
4,0.325625,"POLYGON ((9504300.8 3283117.61, 9504300.8 3283...",249.480,False
5,0.120144,"POLYGON ((9504164 3283182.11, 9504164 3283161....",324.450,False
6,0.763121,"POLYGON ((9504089.9 3283098.71, 9504109.4 3283...",540.135,False
7,0.638633,"POLYGON ((9504084.2 3283183.61, 9504121.7 3283...",635.760,False
8,0.853731,"POLYGON ((9504095.6 3283219.31, 9504141.5 3283...",1224.765,False


In [424]:
df_post

,confidence,geometry


In [426]:
# df_pre  = get_buildings(ds_pre,google=False)
# df_post = get_buildings(ds_comp, google=False)

def on_water(df, mask):
    if not len(df): return np.zeros(0, bool)
    p = df['geometry'].centroid
    return mask.sel(x=xr.DataArray(p.x, dims='p'), y=xr.DataArray(p.y, dims='p'),
                    method='nearest').values.astype(bool)

if len(df_pre):  df_pre  = df_pre[~on_water(df_pre, ds_water_dino_pre)].reset_index(drop=True)
if len(df_post): df_post = df_post[~on_water(df_post, ds_water_dino_pre)].reset_index(drop=True)

# a pre-flood building SURVIVES if any post detection lands within 10 m of it
if len(df_pre) and len(df_post):
    near = gpd.sjoin_nearest(df_pre, df_post[['geometry']], how='inner',
                             max_distance=10, distance_col='dist')
    survived = set(near.index.unique())
else:
    survived = set()

df_pre['survived'] = df_pre.index.isin(survived) if len(df_pre) else False
n_survived = int(df_pre['survived'].sum()) if len(df_pre) else 0
n_lost     = len(df_pre) - n_survived
df_post = df_pre[df_pre['survived']].reset_index(drop=True) if len(df_pre) else df_pre

print(f'buildings pre : {len(df_pre)}')
print(f'  survived    : {n_survived}')
print(f'  lost        : {n_lost}  ({n_lost/max(len(df_pre),1)*100:.0f}%)')

buildings pre : 9
  survived    : 0
  lost        : 9  (100%)


In [427]:
opts = dict(x='x', y='y', grid=True, c='confidence', cmap='Category20', alpha=0.5,
            line_color='black', line_width=2, colorbar=False, hover_cols=['confidence'])

def show(ds, df, title):
    img = ds.hvplot.rgb(x='x', y='y', bands='band', height=600, width=600,
                        rasterize=True, robust=True, title=f'{title} : {len(df)} Buildings')
    return img * df.hvplot(**opts) if len(df) else img

(show(ds_pre, df_pre, 'Pre Flood') + show(ds_comp, df_post, 'Post Flood')).cols(2)

BokehModel(combine_events=True, render_bundle={'docs_json': {'659b835c-0845-4df0-85d3-192e436ef0a0': {'version…

<br>**Roads & Bridges : Segments Impacted by Flood Extent Expansion**

In [384]:
cache_r = f'/tmp/osm_road_{lat:.4f}_{lon:.4f}.gpkg'
MAJOR = ['motorway','trunk','primary','secondary','tertiary']

if os.path.exists(cache_r):
    df_road = gpd.read_file(cache_r)
else:
    b = ds_pre.rio.transform_bounds('EPSG:4326')
    sel = '|'.join(MAJOR + [m+'_link' for m in MAJOR])
    q = f'[out:json][timeout:60];way({b[1]},{b[0]},{b[3]},{b[2]})["highway"~"^({sel})$"];out geom;'
    rows = []
    for e in overpass(q):
        pts = [(p['lon'], p['lat']) for p in e.get('geometry', [])]
        if len(pts) < 2: continue
        t = e.get('tags', {})
        rows.append({'geometry':LineString(pts), 'name':t.get('name'), 'highway':t.get('highway'),
                     'kind':'bridge' if t.get('bridge') else 'road'})
    df_road = gpd.GeoDataFrame(rows, columns=['geometry','name','highway','kind'], crs='epsg:4326')
    if len(df_road): df_road.to_file(cache_r, driver='GPKG')

if len(df_road):
    df_road = df_road.to_crs(ds_comp.rio.crs)
    tile = box(*ds_comp.rio.bounds())
    df_road['geometry'] = df_road.intersection(tile)
    df_road = df_road[~df_road.is_empty & df_road.geometry.notna()].reset_index(drop=True)

print(f"roads {len(df_road)}")

roads 16   bridges 1   {'service': 7, 'primary': 3, 'track': 2, 'unclassified': 2, 'residential': 2, 'path': 1}


In [385]:
flood = ds_water_dino.rio.reproject_match(ds_comp, resampling=Resampling.nearest)
poly  = gpd.GeoSeries(
    [shape(g) for g, v in shapes(flood.values.astype('uint8'), transform=flood.rio.transform()) if v == 1],
    crs=ds_comp.rio.crs).union_all()

df_road['length_m']  = df_road['geometry'].length
df_road['flooded_m'] = df_road['geometry'].intersection(poly).length      # only the submerged part
df_road['frac']      = df_road['flooded_m'] / df_road['length_m'].clip(lower=1e-9)
df_road['impacted']  = df_road['flooded_m'] > 0

is_br  = df_road['kind'] == 'bridge'
n_br, n_br_imp = int(is_br.sum()), int((is_br & df_road['impacted']).sum())
km_tot = df_road.loc[~is_br, 'length_m'].sum()/1000
km_imp = df_road.loc[~is_br, 'flooded_m'].sum()/1000                      # sum of flooded lengths
n_imp  = int(df_road['impacted'].sum())

cols = ['name','kind','highway','length_m','flooded_m','frac']
print(df_road[df_road['impacted']][cols].sort_values('flooded_m', ascending=False).to_string(index=False)
      if n_imp else 'no road intersects flood extent')

                name   kind      highway   length_m  flooded_m     frac
Pasang Lhamu Highway   road      primary 310.483597 299.808787 0.965619
                 NaN   road      service  91.703791  91.703791 1.000000
                 NaN   road      service  90.195551  80.594026 0.893548
                 NaN   road      service  80.559235  80.559235 1.000000
Pasang Lhamu Highway   road      primary 136.885957  76.573195 0.559394
                 NaN   road  residential  76.256774  76.256774 1.000000
                 NaN   road        track  57.075116  57.075116 1.000000
                 NaN   road      service  39.698015  39.698015 1.000000
                 NaN   road unclassified 349.795304  33.716979 0.096391
Pasang Lhamu Highway bridge      primary  24.691660  24.691660 1.000000
                 NaN   road unclassified  20.945795  20.945795 1.000000
                 NaN   road  residential   3.574909   3.574909 1.000000


In [386]:
title = f'Flooded : {km_imp:.2f} km of {km_tot:.2f} km road, {n_br_imp}/{n_br} bridge'

p = ds_comp.hvplot.rgb(x='x',y='y',bands='band',height=650,width=700,rasterize=True,robust=True,title=title)*\
    ds_water_dino.where(ds_water_dino>0).hvplot(x='x',y='y',rasterize=True,cmap=['red'],colorbar=False,alpha=0.3)*\
    df_road.hvplot.paths(color='white', line_width=2, hover_cols=['name','kind','frac'])

cut = gpd.GeoDataFrame(df_road[df_road['impacted']].drop(columns='geometry'),
                       geometry=df_road[df_road['impacted']]['geometry'].intersection(poly),
                       crs=df_road.crs)
cut = cut[~cut.is_empty]
if len(cut): p = p * cut.hvplot.paths(color='yellow', line_width=5, hover_cols=['name','kind','frac'])
p

BokehModel(combine_events=True, render_bundle={'docs_json': {'04bd85dd-1c6c-4af7-9e67-4c7611e0cfcc': {'version…

<br>**Critical Infrastructure : Hydropower, Power, Health, Shelter, Settlements**

In [397]:
cache_i = f'/tmp/osm_infra_v2_{lat:.4f}_{lon:.4f}.gpkg'

TAGS = {
    'power=plant':'hydropower', 'power=generator':'hydropower',
    'waterway=dam':'hydropower', 'waterway=weir':'hydropower', 'man_made=pipeline':'hydropower',
    'power=substation':'electricity', 'power=line':'electricity', 'power=tower':'electricity',
    'amenity=hospital':'healthcare', 'amenity=clinic':'healthcare',
    'amenity=school':'shelter', 'amenity=place_of_worship':'shelter',
    'aeroway=helipad':'air_access',
    'place=village':'settlement', 'place=hamlet':'settlement', 'place=town':'settlement',
}

if os.path.exists(cache_i) and 'category' in gpd.read_file(cache_i, rows=1).columns:
    df_infra = gpd.read_file(cache_i)
else:
    b = ds_pre.rio.transform_bounds('EPSG:4326')
    bb = f'{b[1]},{b[0]},{b[3]},{b[2]}'
    q = ('[out:json][timeout:60];('
         f'nwr({bb})["power"~"plant|generator|substation|line|tower"];'
         f'nwr({bb})["waterway"~"dam|weir"];'
         f'nwr({bb})["man_made"="pipeline"];'
         f'nwr({bb})["amenity"~"hospital|clinic|school|place_of_worship"];'
         f'nwr({bb})["place"~"village|hamlet|town"];'
         f'nwr({bb})["aeroway"="helipad"];);out geom center;')
    rows = []
    for e in overpass(q):
        t = e.get('tags', {})
        kv = next((f'{k}={t[k]}' for k in ('power','waterway','man_made','amenity','place','aeroway')
                   if t.get(k) and f'{k}={t[k]}' in TAGS), None)
        if kv is None: continue
        if 'geometry' in e:
            pts = [(p['lon'], p['lat']) for p in e['geometry']]
            if len(pts) < 2: continue
            g = Polygon(pts) if (pts[0]==pts[-1] and len(pts)>3) else LineString(pts)
        elif 'center' in e: g = Point(e['center']['lon'], e['center']['lat'])
        elif 'lon' in e:    g = Point(e['lon'], e['lat'])
        else: continue
        rows.append({'geometry':g, 'name':t.get('name'), 'category':TAGS[kv], 'osm_tag':kv})

    df_infra = gpd.GeoDataFrame(rows, columns=['geometry','name','category','osm_tag'], crs='epsg:4326')
    if len(df_infra): df_infra.to_file(cache_i, driver='GPKG')

df_infra = df_infra.to_crs(ds_comp.rio.crs)
tile = box(*ds_comp.rio.bounds())
df_infra = df_infra[df_infra.intersects(tile)].reset_index(drop=True)
df_infra['geometry'] = df_infra.intersection(tile)                         # clip to tile
df_infra = df_infra[~df_infra.is_empty & df_infra.geometry.notna()].reset_index(drop=True)
print(f'{len(df_infra)} features in tile')
print(df_infra['category'].value_counts().to_string())


2 features in tile
category
electricity    1
shelter        1


In [398]:
df_infra['impacted'] = df_infra.intersects(poly) | (df_infra.distance(poly) <= 50)

stats = (df_infra.groupby('category')['impacted'].agg(total='size', impacted='sum')
                 .sort_values('impacted', ascending=False))
print(stats.to_string())
print()
hit = df_infra[df_infra['impacted']]
print(hit[['name','category','osm_tag']].to_string(index=False) if len(hit) else 'no infrastructure impacted')

n_hydro   = int(hit['category'].isin(['hydropower']).sum())
n_power   = int(hit['category'].isin(['electricity']).sum())
n_health  = int(hit['category'].isin(['healthcare']).sum())
n_shelter = int(hit['category'].isin(['shelter']).sum())
n_air     = int(hit['category'].isin(['air_access']).sum())
n_settl   = int(hit['category'].isin(['settlement']).sum())

             total  impacted
category                    
electricity      1         1
shelter          1         1

                     name    category        osm_tag
                      NaN electricity    power=tower
Shree Ramchandra Ni Ma Vi     shelter amenity=school


In [399]:
q = ds_comp.hvplot.rgb(x='x',y='y',bands='band',height=650,width=700,rasterize=True,robust=True,
                       title=f'Infrastructure Impacted : {n_hydro} hydro, {n_power} power, {n_health} health, {n_shelter} shelter, {n_settl} village')*\
    ds_water_dino.where(ds_water_dino>0).hvplot(x='x',y='y',rasterize=True,cmap=['red'],colorbar=False,alpha=0.3)

for flag, col, sz in [(False,'white',80), (True,'yellow',220)]:
    sub = df_infra[df_infra['impacted']==flag]
    pts = sub[sub.geom_type=='Point']
    lns = sub[sub.geom_type!='Point']
    if len(pts): q = q * pts.hvplot.points(color=col, size=sz, line_color='black', hover_cols=['name','category','osm_tag'])
    if len(lns): q = q * lns.hvplot.paths(color=col, line_width=3, hover_cols=['name','category','osm_tag'])
q

BokehModel(combine_events=True, render_bundle={'docs_json': {'cbf0bf62-b693-491a-8556-fb19ed4471fe': {'version…

In [400]:
px_d = abs(float(sim.x[1]-sim.x[0])) * abs(float(sim.y[1]-sim.y[0]))

print(f'{"="*46}')
print(f'  NEPAL FLOOD 2026 - DAMAGE ASSESSMENT')
print(f'  {lat:.4f}, {lon:.4f}   {size*res/1000:.2f} x {size*res/1000:.2f} km tile')
print(f'{"="*46}')
print(f'  FLOOD EXTENT')
print(f'    pre-flood water   : {float(ds_water_dino_pre.sum())*px_d/1e6:8.4f} km2')
print(f'    post-flood water  : {float(ds_water_dino.sum())*px_d/1e6:8.4f} km2')
print(f'    expansion         : {float(ds_water_dino.sum())/max(float(ds_water_dino_pre.sum()),1):8.2f} x')
print()
print(f'  BUILDINGS')
print(f'    pre-flood         : {len(df_pre):8d}')
print(f'    surviving         : {len(df_post):8d}')
print(f'    lost              : {len(df_pre)-len(df_post):8d}   ({(1-len(df_post)/max(len(df_pre),1))*100:.0f}%)')
print()
print(f'  ROADS')
print(f'    total             : {km_tot:8.2f} km')
print(f'    impacted          : {km_imp:8.2f} km   ({km_imp/max(km_tot,1e-9)*100:.0f}%)')
print()
print(f'  BRIDGES')
print(f'    total             : {n_br:8d}')
print(f'    impacted          : {n_br_imp:8d}')
print()
print(f'  INFRASTRUCTURE')
print(f'    hydropower        : {n_hydro:8d} impacted')
print(f'    electricity       : {n_power:8d} impacted')
print(f'    healthcare        : {n_health:8d} impacted')
print(f'    shelter (school/temple) : {n_shelter:d} impacted')
print(f'    air access        : {n_air:8d} impacted')
print(f'    settlements       : {n_settl:8d} impacted')
print(f'{"="*46}')

  NEPAL FLOOD 2026 - DAMAGE ASSESSMENT
  27.9700, 85.1830   0.61 x 0.61 km tile
  FLOOD EXTENT
    pre-flood water   :   0.0931 km2
    post-flood water  :   0.3011 km2
    expansion         :     3.23 x

  BUILDINGS
    pre-flood         :       45
    surviving         :       18
    lost              :       27   (60%)

  ROADS
    total             :     1.61 km
    impacted          :     0.86 km   (53%)

  BRIDGES
    total             :        1
    impacted          :        1

  INFRASTRUCTURE
    hydropower        :        0 impacted
    electricity       :        1 impacted
    healthcare        :        0 impacted
    shelter (school/temple) : 1 impacted
    air access        :        0 impacted
    settlements       :        0 impacted
